In [1]:
# install dependencies
!pip install chromadb sentence-transformers groq -q

In [ ]:
import zipfile
import chromadb
import re
from sentence_transformers import SentenceTransformer
from groq import Groq

# Extract chroma_db - vector database with pre-computed embeddings
with zipfile.ZipFile('Files.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
# Initialize embedding model and load ChromaDB
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Two separate collections: global (totals) and grouped (trends/promotions)
client = chromadb.PersistentClient(path="./chroma_db")
collection_global = client.get_or_create_collection(name="global_summaries")
collection_grouped = client.get_or_create_collection(name="grouped_summaries")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Initialize GROQ client 
GROQ_API_KEY = "your_groq_api_key_here"
client_groq = Groq(api_key=GROQ_API_KEY)

In [ ]:
def extract_amount(text):
    # Extract dollar values for sorting by sales/profit
    match = re.search(r'\$[\d,]+\.?\d*', text)
    return float(match.group().replace('$', '').replace(',', '')) if match else 0

def extract_margin(text):
    # Extract margin percentages for sorting by profitability
    match = re.search(r'Margin\s+([-\d.]+)%', text)
    return float(match.group(1)) if match else 0

def extract_discount(text):
    # Extract discount rates for identifying promotional items
    match = re.search(r'Average discount (\d+)%', text)
    return float(match.group(1)) if match else 0

In [ ]:
def retrieve_context(query, num_results=20):
    # Special handling for discount/promotional queries - retrieve from grouped collection
    if "discount" in query.lower() or "frequently sold" in query.lower():
        grouped_results = collection_grouped.query(
            query_texts=["frequently discounted"],
            n_results=2000,
            include=["documents", "metadatas"])

        documents = []
        if grouped_results['documents'] and len(grouped_results['documents']) > 0:
            for doc in grouped_results['documents'][0]:
                if "Frequently discounted product" in doc:
                    documents.append(doc)

        documents = sorted(documents, key=extract_discount, reverse=True)
        documents = documents[:num_results]
        context = "\n".join(documents)
        return context[:2500] if len(context) > 2500 else context

    # Standard retrieval - query both global and grouped collections
    global_results = collection_global.query(
        query_texts=[query],
        n_results=num_results,
        include=["documents", "metadatas"]
    )

    grouped_results = collection_grouped.query(
        query_texts=[query],
        n_results=num_results,
        include=["documents", "metadatas"])

    documents = []

    # Filter global results - only aggregate metrics
    if global_results['documents'] and len(global_results['documents']) > 0:
        for doc, meta in zip(global_results['documents'][0], global_results['metadatas'][0]):
            if not meta.get('is_aggregate', True):
                continue
            documents.append(doc)

    # Include all grouped results
    if grouped_results['documents'] and len(grouped_results['documents']) > 0:
        for doc, meta in zip(grouped_results['documents'][0], grouped_results['metadatas'][0]):
            documents.append(doc)

    # Sort by metric type - margin queries sorted by percentage, others by amount
    if "margin" in query.lower() and ("sub-categ" in query.lower() or "category" in query.lower()):
        documents = sorted(documents, key=extract_margin, reverse=True)
    else:
        documents = sorted(documents, key=extract_amount, reverse=True)

    documents = documents[:num_results]
    context = "\n".join(documents)
    return context[:2500] if len(context) > 2500 else context

In [ ]:
def detect_answer_mode(query):
    # Detect query type to apply appropriate response template
    q = query.lower()

    if "cities" in q and "top performers" in q:
        return "city_profit_ranking"

    if "compare" in q:
        return "comparison"

    if "trend" in q or "over time" in q:
        return "trend"

    if "top" in q or "highest" in q or "best" in q:
        return "ranking"

    if "which" in q:
        return "selection"

    return "default"

# Response templates - enforce answer format based on query type
templates = {
"trend": """
List all periods chronologically with exact values.
Then give one short factual trend statement
based only on those values.
No interpretation beyond the data.
""",

"ranking": """
Use only the ranked entities explicitly present in context.
Do not infer alternate ranking metrics.
State the top performer first.
Then sort the ranking in the order of performance.
""",

"comparison": """
State which entity performs better first,
then compare all entities using exact values.
Mention differences when possible.
""",

"selection": """
Answer the question directly first,
then support it with the relevant entities and values.
""",

"city_profit_ranking": """
Rank cities ONLY by absolute profit (not margin).
Return only the top 5 cities.
State the top performer first.
Then sort the ranking in the order of profit.
Use city names and profit values exactly from context.
Do not rank by margin percentage.
"""
}

In [ ]:
def rag_query(query):
    # Full RAG pipeline: retrieve context -> detect mode -> generate factual answer
    context = retrieve_context(query, 20)

    if not context:
        return "Data not provided."

    # Apply appropriate response template based on query type
    mode = detect_answer_mode(query)
    task_instruction = templates[mode]

    prompt = f"""
    Context:
    {context}

    Question:
    {query}

    Task:
    {task_instruction}

    Rules:
    1. Answer the question directly in the first sentence.

    2. Use only facts explicitly present in the context.

    3. Use the metric requested in the question only
      (do not switch sales, profit, or margin).

    4. Preserve labels, values, and ranking exactly as supported
      by the context.

    5. Do not infer missing facts or add interpretations
      beyond the data.

    6. If returning a ranked or comparative list,
      each entity must appear once only.
    """

    # Call LLM with zero temperature for deterministic factual answers
    response = client_groq.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a data assistant. Never use thinking blocks. Answer directly and factually. If multiple items are shown, list them all with their full names and values."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        model="qwen/qwen3-32b",
        temperature=0.0,
        max_tokens=300
    )

    answer = response.choices[0].message.content.strip()

    # Clean up unwanted patterns
    if "<think>" in answer:
        answer = answer.split("</think>")[-1].strip()

    answer = answer.replace("Okay, let's see.", "").replace("let's see.", "").strip()

    return answer

In [ ]:
# Test section - verify RAG accuracy across different query types
test_queries = [
    "What is the sales trend over the 4-year period?",
    "Which months show the highest sales? Is there seasonality?",
    "How has profit margin changed over time?",
    "Which product category generates the most revenue?",
    "What sub-categories have the highest profit margins?",
    "Which products are frequently sold at a discount?",
    "Which region has the best sales performance?",
    "Compare sales performance across different states.",
    "Which cities are the top performers?",
    "Compare Technology vs Furniture sales trends.",
    "How does the West region compare to the East in terms of profit?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\nQ{i}: {query}")
    print("-" * 80)
    print(f"A: {rag_query(query)}\n")


Q1: What is the sales trend over the 4-year period?
--------------------------------------------------------------------------------
A: The sales trend over the 4-year period is as follows:  
2014: $484,247.50  
2015: $470,532.51  
2016: $609,205.60  
2017: $733,215.26  

Sales increased from 2014 to 2017.


Q2: Which months show the highest sales? Is there seasonality?
--------------------------------------------------------------------------------
A: The months with the highest sales are November, December, and September, and there is high seasonality associated with these months.  

1. November: $352,461.07  
2. December: $325,293.50  
3. September: $307,649.95


Q3: How has profit margin changed over time?
--------------------------------------------------------------------------------
A: Profit margin has changed over time as follows:  

- Year 2014 TOTAL: Margin 11.81%  
- Year 2015 TOTAL: Margin 11.76%  
- Year 2016 TOTAL: Margin 12.98%  
- Year 2017 TOTAL: Margin 11.60%  

Tre

In [ ]:
def interactive_mode():
    # Interactive RAG - freeform questions with natural language responses
    print("Interactive RAG Mode - Type 'quit' to exit\n")
    while True:
        try:
            query = input(">>> Your question: ").strip()
            if query.lower() == 'quit':
                break
            if query:
                try:
                    print(f"\n{rag_query(query)}\n")
                except Exception:
                    print("\nNo answer found in the data.\n")
        except KeyboardInterrupt:
            break

# Uncomment to run
#interactive_mode()